In [1]:
# Imports 
import os
import sys
import glob
import math as m
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rc
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


from scipy import interpolate
from scipy.interpolate import griddata, interp1d
from scipy.integrate import trapezoid

from concurrent.futures import ProcessPoolExecutor, as_completed

plt.rcParams.update(plt.rcParamsDefault)

sys.path.extend([
    '/projects/DEIKE/cmartinb/jupyter_notebook/project_specific/turbulence',
    '/projects/DEIKE/cmartinb/functions',
])

os.chdir('/projects/DEIKE/cmartinb/')

from funciones import *

print(os.environ['PATH'])

/usr/licensed/anaconda3/2024.6/bin:/usr/licensed/anaconda3/2024.6/condabin:/home/cm6797/.local/bin:/home/cm6797/bin:/usr/share/Modules/bin:/usr/local/bin:/usr/local/sbin:/usr/bin:/usr/sbin:/opt/puppetlabs/bin:/opt/dell/srvadmin/bin


In [1]:
def interp_2d(xdata,zdata,xtile,ztile,fld):
    #
    fld_int = griddata((xdata.ravel(), zdata.ravel()), fld.ravel(), (xtile, ztile), method='nearest');
    #
    return fld_int;
#
def from_matrix(pfile,N):
    #
    snapshot = np.fromfile(pfile, dtype=np.float64);
    snapshot = snapshot.reshape([N,N]);
    snapshot = np.transpose(snapshot);
    #
    return snapshot
#
def keep_center_colormap(colMap, vmin, vmax, center):
    #
    vmin = vmin - center
    vmax = vmax - center
    dv = max(-vmin, vmax) * 2
    N = int(256 * dv / (vmax-vmin))
    colMap_str = cm1.get_cmap(colMap, N)
    newcolors = colMap_str(np.linspace(0, 1, N))
    beg = int((dv / 2 + vmin)*N / dv)
    end = N - int((dv / 2 - vmax)*N / dv)
    newmap = ListedColormap(newcolors[beg:end])
    #
    return newmap
#
def get_int_qtn(work_dir,eta_files,index,N,L0):
    #
    tot_col = 18;
    istep = eta_files[index][-13:-4];
    print(istep)
    etalo = np.fromfile(work_dir+'eta/eta_loc/eta_loc_p3_t'+istep+'.bin')
    size  = etalo.shape;
    tot_row_i = int(size[0]/tot_col);
    etalo = etalo.reshape([tot_row_i, tot_col]);
    #
    # we remove
    #
    print("First pass of remove")
    eta_m0  = 1.0; cirp_th = 0.175;
    new_row = 0;
    for i in range(tot_row_i):
        if ( abs(etalo[i][12]-eta_m0) < cirp_th ):
           new_row += 1;
    #
    print("Second pass of remove")
    etal = np.zeros([new_row, tot_col]);
    for i in range(new_row):
        if ( abs(etalo[i][12]-eta_m0) < cirp_th ):
           etal[i][:] = etalo[i][:];
    #
    print("Assign array")
    xpo = etal[:,0]; zpo = etal[:,1];
    uxx = etal[:,2]; uyy = etal[:,3]; uzz = etal[:,4];
    Sxx = etal[:,5]; Syy = etal[:,6]; Szz = etal[:,7];
    Sxy = etal[:,8]; Sxz = etal[:,9]; Syz = etal[:,10];
    pre = etal[:,11];
    eta = etal[:,12]; 
    nxx = etal[:,14]; nyy = etal[:,15]; nzz = etal[:,16];
    #
    # We interpolate eta on a 2D Cartesian grid
    # with equidistant spacing equal to the printing resolution (2**9)
    #
    print("Interpolation to a Cartesian grid")
    #
    uxx_int = interp_2d(xpo, zpo, x_til, z_til, uxx);
    uyy_int = interp_2d(xpo, zpo, x_til, z_til, uyy);
    uzz_int = interp_2d(xpo, zpo, x_til, z_til, uzz);
    #
    pre_int = interp_2d(xpo, zpo, x_til, z_til, pre);
    #
    Sxx_int = interp_2d(xpo, zpo, x_til, z_til, Sxx);
    Syy_int = interp_2d(xpo, zpo, x_til, z_til, Syy);
    Szz_int = interp_2d(xpo, zpo, x_til, z_til, Szz);
    Sxy_int = interp_2d(xpo, zpo, x_til, z_til, Sxy);
    Sxz_int = interp_2d(xpo, zpo, x_til, z_til, Sxz);
    Syz_int = interp_2d(xpo, zpo, x_til, z_til, Syz);
    #
    eta_int = interp_2d(xpo, zpo, x_til, z_til, eta);
    nxx_int = interp_2d(xpo, zpo, x_til, z_til, nxx);
    nyy_int = interp_2d(xpo, zpo, x_til, z_til, nyy);
    nzz_int = interp_2d(xpo, zpo, x_til, z_til, nzz);
    #
    # Normalize the normal vector
    #
    norm = np.sqrt(nxx_int**2+nyy_int**2+nzz_int**2)
    nxx_int = nxx_int/norm; nyy_int = nyy_int/norm; nzz_int = nzz_int/norm; 
    #
    # Compute u',v',w'
    #
    uxx_int = uxx_int-np.average(uxx_int);
    uyy_int = uyy_int-np.average(uyy_int);
    uzz_int = uzz_int-np.average(uzz_int);
    #
    eta_int = eta_int - np.average(eta_int);
    pre_int = pre_int - np.average(pre_int);
    uun_int = (
                (uxx_int) * nxx_int +
                (uyy_int) * nyy_int +
                (uzz_int) * nzz_int
              );
    #
    # Compute the traction vector
    #
    tnx_int = Sxx_int*nxx_int+Sxy_int*nyy_int++Sxz_int*nzz_int;
    tny_int = Sxy_int*nxx_int+Syy_int*nyy_int++Syz_int*nzz_int;
    tnz_int = Sxz_int*nxx_int+Syz_int*nyy_int++Szz_int*nzz_int;
    #
    return eta_int,pre_int,uun_int,tnx_int,tny_int,tnz_int,uxx_int,uyy_int,uzz_int;
    #
#
def spec_onevar(var, L, N):
    #
    spectrum = np.fft.fft2(var) / N**2
    G = spectrum * np.conjugate(spectrum);
    F = np.absolute(G);
    #
    return F;
#
def spec_twovar(var1, var2, L, N, ind):
    #
    spectrum1 = np.fft.fft2(var1) / N**2
    spectrum2 = np.fft.fft2(var2) / N**2
    G = spectrum1 * np.conjugate(spectrum2)
    if(  ind == 0):
      F = np.absolute(G);
    elif(ind == 1):
      F = np.real(G);
    elif(ind == 2):
      F = np.imag(G);
    else:
      print("Wrong choice");
      F = np.nan;
    #
    return F;
#
def spec_integr(F, L, N, CHECK=False):
    #
    if CHECK: print (np.sum(F))
    wavenumber = 2*np.pi*np.fft.fftfreq(n=N,d=L/N)
    kx = np.fft.fftshift(wavenumber); 
    ky = kx
    kx_tile, ky_tile = np.meshgrid(kx,ky)
    theta = np.arange(-N/2,N/2)/(N)*2*np.pi
    k = wavenumber[0:int(N/2)]
    dkx = kx[1] - kx[0]; dky = ky[1] - ky[0]
    dk = k[1]-k[0]; dtheta = theta[1]-theta[0]
    k_tile, theta_tile = np.meshgrid(k,theta)
    kxp_tile, kyp_tile = pol2cart(k_tile, theta_tile)
    #
    F_center = np.fft.fftshift(F)/dkx/dky # Further normalization by independent variables
    F_center_polar = griddata((kx_tile.ravel(),ky_tile.ravel()), F_center.ravel(), (kxp_tile, kyp_tile), method='nearest')
    F_cpi = np.sum(F_center_polar*k_tile, axis=0)*dtheta # Azimuthal integration
    if CHECK: print (np.sum(F_cpi)*dk)
    #
    return F_cpi;
#
def cart2pol(x, y):
    rho = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return(rho, phi)
#
def pol2cart(rho, phi):
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return(x, y)

In [3]:
# -------------------------------------------------------------------
# USER PATHS
# -------------------------------------------------------------------
#work_dir = "/scratch/cimes/ns8802/my_broadband/re720_bo0200_kpHs0.02_uoc0p50_reW1.0e5_L10/"
#work_dir = "/scratch/cimes/ns8802/my_broadband/re720_bo0200_kpHs0.16_uoc0p50_reW2.0e4_L10/"
work_dir = "/projects/DEIKE/nscapin/re720_bo0200_kpHs0.16_uoc0p50_reW2.5e4_L10/"

save_dir = "/projects/DEIKE/cmartinb/notebooks/pressure/"
out_dir  = os.path.join(save_dir, "postproc_saved")
os.makedirs(out_dir, exist_ok=True)

# -------------------------------------------------------------------
# INPUT FILES
# -------------------------------------------------------------------
eta_files = np.sort(glob.glob(work_dir + "/eta/eta_loc/eta_loc_p3_t*"))
time_eta  = pd.read_csv(work_dir + "eta/global_int_3.out", header=None, sep=" ").to_numpy()

# GRID 
N  = 512
L0 = 2.0*np.pi
kp = 2.0*np.pi/(L0/4.0)

x_int = np.linspace(-L0/2, L0/2, N, endpoint=False) + L0/N/2
z_int = np.linspace(-L0/2, L0/2, N, endpoint=False) + L0/N/2
x_til, z_til = np.meshgrid(x_int, z_int)

ext_1 = 10
st    = 10

n_time = int(len(time_eta))         
n_etaT = int(len(eta_files) // 2)
n_tot  = min(n_time, n_etaT)

if ext_1 < 0:
    ext_1 = 0
if ext_1 >= n_tot - 1:
    raise ValueError(f"ext_1={ext_1} is too large for n_tot={n_tot}")


ext_2 = ext_1 + (n_tot - ext_1)//2
ext_2 = max(ext_2, ext_1 + 1)  
print("====================================")
print("work_dir:", work_dir)
print(f"n_time={n_time}, n_eta_files={len(eta_files)}, n_etaT(=len/3)={n_etaT}, n_tot={n_tot}")
print(f"PROCESSING ONLY FIRST HALF: ext_1={ext_1}, ext_2={ext_2}, st={st}")
print("====================================")

# --------------------
# PHYSICAL PARAMETERS
# --------------------
rho_a = 1.225/1000
rho_w = 1.0
UstarRATIO = 0.50
beta = 30.0
u_ast = 0.25
cp = u_ast / UstarRATIO
g_ = kp * cp**2
mu_a = rho_a*u_ast*(2.0*np.pi - 1.0)/720.0
lambda_p = 2*np.pi/4
nul = cp * lambda_p / 1e5

sigma = 0.072

print("nul, cp:", nul, cp)
print("sigma:", sigma)


omega_p = np.sqrt(g_ * kp + (sigma / rho_w) * kp**3)
Tp      = 2.0*np.pi / omega_p
t_over_Tp_all = (time_eta[:, 0] - time_eta[0, 0]) / Tp

print("omega_p, Tp:", omega_p, Tp)


loop_inds = list(range(ext_1, ext_2, st))
if len(loop_inds) == 0:
    raise ValueError("loop_inds is empty: check ext_1/ext_2/st")

map_cm_e = plt.get_cmap("viridis")
map_cm_p = plt.get_cmap("viridis")
map_cm_t = plt.get_cmap("plasma")

t_st = float(t_over_Tp_all[loop_inds[0]])
t_in = 0.0
t_fi = float(t_over_Tp_all[loop_inds[-1]] - t_st)

norm_time = mpl.colors.Normalize(vmin=t_in, vmax=t_fi)

t_loop = np.array([float(t_over_Tp_all[i] - t_st) for i in loop_inds], dtype=float)
colors_e = map_cm_e(norm_time(t_loop))
colors_p = map_cm_p(norm_time(t_loop))
colors_t = map_cm_t(norm_time(t_loop))


work_dir: /projects/DEIKE/nscapin/re720_bo0200_kpHs0.16_uoc0p50_reW2.5e4_L10/
n_time=436, n_eta_files=436, n_etaT(=len/3)=218, n_tot=218
PROCESSING ONLY FIRST HALF: ext_1=10, ext_2=114, st=10
nul, cp: 7.853981633974482e-06 0.5
sigma: 0.072
omega_p, Tp: 2.93393933134276 2.141552567245484


In [5]:
"""
PARALLEL version (windows in parallel)

Loop over ALL outputs but grouped into windows:
  - WIN = 150 outputs per window
  - overlap = 50%  => STRIDE = 75

Saving:
  - information for each window (times, indices)
  - integrals Sp, St, Se
  - window-averaged spectra (npz)
"""

work_dir = "/projects/DEIKE/nscapin/re720_bo0200_kpHs0.16_uoc0p50_reW2.5e4_L10/"
save_dir = "/projects/DEIKE/cmartinb/notebooks/pressure/"
os.makedirs(save_dir, exist_ok=True)

eta_files = np.sort(glob.glob(os.path.join(work_dir, "eta/eta_loc/eta_loc_p3_t*")))
time_eta  = pd.read_csv(os.path.join(work_dir, "eta/global_int_3.out"),
                        header=None, sep=" ").to_numpy()

# GRID 
N  = 512
L0 = 2.0*np.pi
kp = 2.0*np.pi/(L0/4.0)

# WINDOWING

WIN     = 75
OVERLAP = 0.50
STRIDE  = int(WIN * (1.0 - OVERLAP))   # 75

ext_1 = 0
ext_2 = min(len(time_eta), len(eta_files))

if ext_2 - ext_1 < WIN:
    raise ValueError(f"No hay suficientes outputs: ext_2-ext_1={ext_2-ext_1} < WIN={WIN}")

# -------------
# PHYS PARAMS 
# -------------
rho_a = 1.225/1000
rho_w = 1.0
UstarRATIO = 0.50
beta = 30.0
u_ast = 0.25
cp = u_ast / UstarRATIO

g = kp * cp**2
mu_a = rho_a*u_ast*(2.0*np.pi-1.0)/720.0

lambda_p = 2*np.pi/4
nul = cp*lambda_p/10**5

print("work_dir =", work_dir)
print("save_dir =", save_dir)
print("nul, cp  =", nul, cp)

# -------------
# Tp and t/Tp
# -------------
omega_p = np.sqrt(g * kp)
Tp = 2.0*np.pi / omega_p
t_ref = time_eta[0, 0]

windows = []
wcount = 0
print("\n========== WINDOWS (t/Tp) ==========")

for i0 in range(ext_1, ext_2 - WIN + 1, STRIDE):
    i1 = i0 + WIN  # exclusivo
    t0 = time_eta[i0, 0]
    t1 = time_eta[i1 - 1, 0]
    tm = 0.5*(t0 + t1)

    print(f"[win {wcount:03d}] idx {i0:5d}:{i1:5d}   "
          f"t/Tp = {(t0-t_ref)/Tp:9.3f} -> {(t1-t_ref)/Tp:9.3f}   (mid {(tm-t_ref)/Tp:9.3f})")

    windows.append((wcount, i0, i1))
    wcount += 1

print(f"Total windows: {len(windows)}   (WIN={WIN}, STRIDE={STRIDE}, overlap={OVERLAP*100:.0f}%)")
print("====================================\n")


# ==============================
# WORKER (runs in each process)
# ==============================
def process_one_window(args):
    """
    Compute window-averaged spectra + integrals, then save:
      - window_info_{w:03d}.csv
      - sums_window_{w:03d}.csv
      - spectra_window_{w:03d}.npz

    Returns: (w, Sp, St, Se)
    """
    w, i0, i1 = args

    # local times
    t0 = time_eta[i0, 0]
    t1 = time_eta[i1-1, 0]
    tm = 0.5*(t0 + t1)

    # Precompute k 
    wavenumber = 2.0*np.pi*np.fft.fftfreq(n=N, d=L0/N)
    k = wavenumber[0:int(N/2)]

    phi_e_acc = None
    phi_p_acc = None
    phi_t_acc = None
    count = 0

    for ind in range(i0, i1):
        # time + istep
        # time  = time_eta[ind, 0]
        # istep = int(time_eta[ind, 1])

        # upload files
        eta_int, pre_int, uun_int, tnx_int, tny_int, tnz_int, uxx_int, uyy_int, uzz_int = \
            get_int_qtn(work_dir, eta_files, ind, N, L0)

        # spectra
        Phi_e = spec_onevar(eta_int, L0, N)
        phi_e = spec_integr(Phi_e, L0, N, CHECK=True)

        Phi_p = spec_twovar(-pre_int, uun_int, L0, N, 0)
        phi_p = spec_integr(Phi_p, L0, N, CHECK=True)

        Phi_tx = spec_twovar(mu_a*tnx_int, uxx_int, L0, N, 0)
        Phi_ty = spec_twovar(mu_a*tny_int, uyy_int, L0, N, 0)
        Phi_tz = spec_twovar(mu_a*tnz_int, uzz_int, L0, N, 0)
        phi_t  = (spec_integr(Phi_tx, L0, N, CHECK=True)
                + spec_integr(Phi_ty, L0, N, CHECK=True)
                + spec_integr(Phi_tz, L0, N, CHECK=True))

        if phi_e_acc is None:
            phi_e_acc = np.zeros_like(phi_e)
            phi_p_acc = np.zeros_like(phi_p)
            phi_t_acc = np.zeros_like(phi_t)

        phi_e_acc += phi_e
        phi_p_acc += phi_p
        phi_t_acc += phi_t
        count += 1

    # average per window
    phi_e_w = phi_e_acc / count
    phi_p_w = phi_p_acc / count
    phi_t_w = phi_t_acc / count

    # int per window
    Sp = trapezoid(phi_p_w, k)
    St = trapezoid(phi_t_w, k)

    # Se
    g_ = kp*cp**2
    phi_emp = (rho_a/rho_w)*beta*u_ast**2*(k**1.5/g**1.5)*phi_e_w
    Se = (g_*rho_w)*trapezoid(phi_emp, k)

    # --------------------
    # GUARDADO PER WINDOW
    # --------------------
    df_info = pd.DataFrame({
        "i0": [i0],
        "i1": [i1],
        "t0": [t0],
        "t1": [t1],
        "t_mid": [tm],
        "t0_over_Tp": [t0/Tp],
        "t1_over_Tp": [t1/Tp],
        "t_mid_over_Tp": [tm/Tp],
        "WIN": [WIN],
        "STRIDE": [STRIDE]
    })
    fname_info = os.path.join(save_dir, f"window_info_{w:03d}.csv")
    df_info.to_csv(fname_info, index=False)

    df_sums = pd.DataFrame({"Sp": [Sp], "St": [St], "Se": [Se]})
    fname_sums = os.path.join(save_dir, f"sums_window_{w:03d}.csv")
    df_sums.to_csv(fname_sums, index=False)

    fname_spec = os.path.join(save_dir, f"spectra_window_{w:03d}.npz")
    np.savez(fname_spec, k=k, phi_e=phi_e_w, phi_p=phi_p_w, phi_t=phi_t_w)

    return (w, Sp, St, Se)


# ----------------
# MAIN PARALLEL
# ----------------
def main():
    print("========== PROCESSING + SAVING (PARALLEL) ==========")

    n_workers = int(os.environ.get("SLURM_CPUS_PER_TASK", "0")) or os.cpu_count() or 1
    print(f"Using n_workers = {n_workers}")

    results = [None] * len(windows)

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(process_one_window, args): args[0] for args in windows}

        for fut in as_completed(futs):
            w = futs[fut]
            try:
                ww, Sp, St, Se = fut.result()
                results[ww] = (Sp, St, Se)
                print(f"[win {ww:03d}] DONE | Saved window_info/sums/spectra | Sp={Sp:.4e} St={St:.4e} Se={Se:.4e}")
            except Exception as e:
                print(f"[win {w:03d}] FAILED: {e}")
                raise

    Sp_list = np.array([r[0] for r in results])
    St_list = np.array([r[1] for r in results])
    Se_list = np.array([r[2] for r in results])

    df_all = pd.DataFrame({
        "w": np.arange(len(windows)),
        "Sp": Sp_list,
        "St": St_list,
        "Se": Se_list
    })
    fname_all = os.path.join(save_dir, "sums_all_windows.csv")
    df_all.to_csv(fname_all, index=False)

    print("\nDONE. All windows saved in:")
    print(save_dir)
    print(f"Also saved summary: {fname_all}")


if __name__ == "__main__":
    main()


work_dir = /projects/DEIKE/nscapin/re720_bo0200_kpHs0.16_uoc0p50_reW2.5e4_L10/
save_dir = /projects/DEIKE/cmartinb/notebooks/pressure/
nul, cp  = 7.853981633974482e-06 0.5

========== WINDOWS (t/Tp) ==========
[win 000] idx     0:   75   t/Tp =     0.000 ->     4.625   (mid     2.312)
[win 001] idx    37:  112   t/Tp =     2.312 ->     6.937   (mid     4.625)
[win 002] idx    74:  149   t/Tp =     4.625 ->     9.250   (mid     6.937)
[win 003] idx   111:  186   t/Tp =     6.937 ->    11.563   (mid     9.250)
[win 004] idx   148:  223   t/Tp =     9.250 ->    13.875   (mid    11.562)
[win 005] idx   185:  260   t/Tp =    11.563 ->    16.188   (mid    13.875)
[win 006] idx   222:  297   t/Tp =    13.875 ->    18.500   (mid    16.188)
[win 007] idx   259:  334   t/Tp =    16.188 ->    20.812   (mid    18.500)
[win 008] idx   296:  371   t/Tp =    18.500 ->    23.125   (mid    20.812)
[win 009] idx   333:  408   t/Tp =    20.812 ->    25.437   (mid    23.125)
Total windows: 10   (WIN=75, S